In [ ]:
repository_filter: list[str] = []
top_n_methods: int = 200

In [ ]:
from moderne_visualizations_misc.reusable.data_loader import read_data_table
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.express as px
import warnings

warnings.simplefilter("ignore")

# df = read_data_table("../samples/method_quality_metrics.csv")
df = read_data_table("../samples/v2/io.moderne.prethink.table.MethodQualityMetrics.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)
df = qu.add_class_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    df["debtScore"] = qu.compute_debt_score(df)
    top = df.nlargest(min(top_n_methods, len(df)), "debtScore").copy()
    # Fill empty classShort (e.g. JS/TS files without a qualified class name)
    # with the source file name so they still appear in the treemap
    mask = top["classShort"].isna() | (top["classShort"] == "") | (top["classShort"] == "nan")
    if "sourcePath" in top.columns:
        top.loc[mask, "classShort"] = top.loc[mask, "sourcePath"].apply(
            lambda p: p.rsplit("/", 1)[-1] if isinstance(p, str) else "(unknown)"
        )
    else:
        top = top[~mask]
    # Drop any remaining rows that would break the treemap hierarchy
    top = top[
        top["classShort"].notna()
        & (top["classShort"] != "")
        & top["methodName"].notna()
        & (top["methodName"].str.strip() != "")
    ]
    fig = px.treemap(
        top,
        path=["repoShort", "classShort", "methodName"],
        values="lineCount",
        color="debtScore",
        color_continuous_scale=qu.HEALTH_COLORSCALE[::-1],
        hover_data=[
            "cyclomaticComplexity",
            "cognitiveComplexity",
            "maxNestingDepth",
        ],
    )
    fig.update_layout(
        title=f"Technical Debt Treemap (top {len(top)} methods)",
        height=800,
        margin=dict(t=50, l=0, r=0, b=0),
    )
fig.show()